In [ ]:
MODELS = ["smollm2-360m"]
JUDGE = None
PERSONAS = "all"
DIALOGUES = 12
TURNS = 8
REPEATS = 3
LIMIT = 6
BATCH_SIZE = None
RERUN = False

In [ ]:
import gc, hashlib, json, math, os, random, re, string, subprocess, sys, textwrap, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, files, userdata
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate", "pyyaml", "krippendorff"],
                   check=True)
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("no HF_TOKEN secret: downloads work, but more slowly")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from transformers import (AutoModelForCausalLM, AutoModelForImageTextToText, AutoModelForSequenceClassification,
                          AutoTokenizer, BitsAndBytesConfig)

pd.set_option("display.max_colwidth", 120)
DRIVE_ROOT = Path("/content/drive/MyDrive/llm-persona-ei")


def find(name, candidates):
    path = next((path for path in candidates if path.exists()), None)
    if path is None and IN_COLAB:
        print(f"{name} not found: please upload it")
        path = Path(next(iter(files.upload())))
    assert path is not None, f"{name} not found in {[str(c) for c in candidates]}"
    return path


CONFIG_PATH = find("config.yaml", [Path("config.yaml"), Path("../config.yaml"), DRIVE_ROOT / "config.yaml"])
PERSONAS_PATH = find("personas.json", [CONFIG_PATH.parent / "personas.json", DRIVE_ROOT / "personas.json"])
ROOT = Path(os.environ.get("PERSONA_EI_RESULTS") or (DRIVE_ROOT if IN_COLAB else CONFIG_PATH.resolve().parent))

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
for name, path in (("config", CONFIG_PATH), ("personas", PERSONAS_PATH)):
    print(f"{name:9s}:", path.resolve())
print("device   :", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
cfg = yaml.safe_load(CONFIG_PATH.read_text())
model_keys = cfg["tested"] if MODELS == "tested" else list(MODELS)
study_run = set(model_keys) <= set(cfg["tested"])
JUDGE = JUDGE or (cfg["judge"] if study_run else cfg["small"]["judge"])
unknown = [key for key in [*model_keys, JUDGE] if key not in cfg["models"]]
assert not unknown, f"not in config.yaml: {unknown}"
same_family = [key for key in model_keys if cfg["models"][key]["family"] == cfg["models"][JUDGE]["family"]]
assert not (study_run and same_family), f"the judge {JUDGE} is from the same family as {same_family}"

SEED = cfg["seed"]
BATCH_SIZE = BATCH_SIZE or cfg["engine"]["batch_size"]
CHUNK = cfg["engine"]["chunk_size"]
N_BOOTSTRAP = cfg["stats"]["n_bootstrap"]
CHAT = cfg["decoding"]["chat"]
JUDGING = cfg["decoding"]["judge"]
CHECK = cfg["judge_check"]

persona_file = json.loads(PERSONAS_PATH.read_text(encoding="utf-8"))
all_personas = {p["persona_id"]: p for p in persona_file["personas"]}
names = list(all_personas) if PERSONAS == "all" else list(PERSONAS)
personas = {name: f"{all_personas[name]['system_prompt']} {persona_file['shared_suffix']}" for name in names}
TRAITS = ["shy", "formal", "sarcastic", "arrogant", "warm"]
DEFINITIONS = {p["trait"]: p["definition"] for p in persona_file["personas"] if p["trait"]}

OUT_DIR = ROOT / "rq1_persona" / "results"
OUT_DIR = OUT_DIR if study_run else OUT_DIR / "small"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("models  :", ", ".join(model_keys), f"(judge: {JUDGE})")
print("personas:", ", ".join(personas))
print("results :", OUT_DIR, "" if study_run else "(not the study's models, so a folder of its own)")
pd.DataFrame([{"model": key, **{k: cfg["models"][key][k] for k in ("repo", "precision", "params_b")}}
              for key in [*model_keys, JUDGE]])

In [ ]:
from datasets import load_dataset

spec = cfg["rq1"]["conversations"]
dataset = load_dataset(spec["repo"], revision=spec["revision"], split=spec["split"])
candidates, lengths = [], []
for index, row in enumerate(dataset):
    messages = row["utterances"][-1]["history"][0::2]
    lengths.append(len(messages))
    if len(messages) >= TURNS:
        messages = [re.sub(r"\s+([?.!,;:'])", r"\1", message).strip() for message in messages]
        candidates.append({"dialogue_id": f"pc{index:04d}", "user_turns": messages[:TURNS]})
dialogues = sorted(random.Random(SEED).sample(candidates, min(DIALOGUES, len(candidates))),
                   key=lambda dialogue: dialogue["dialogue_id"])

print(f"PersonaChat {spec['split']}: {len(dataset)} dialogues, {len(candidates) / len(dataset):.0%} have {TURNS} or more "
      "messages per speaker")
print("messages per speaker:", pd.Series(lengths).value_counts().sort_index().to_dict())
print("chosen:", [dialogue["dialogue_id"] for dialogue in dialogues])
pd.DataFrame({"turn": range(1, TURNS + 1), f"user message ({dialogues[0]['dialogue_id']})": dialogues[0]["user_turns"]})

In [ ]:
JUDGE_INSTRUCTIONS = (
    "You rate the style of one reply in a conversation. For each of five traits, rate how strongly "
    "the reply shows it: 1 = not at all, 2 = slightly, 3 = moderately, 4 = clearly, 5 = very strongly. "
    "Rate only how the reply is written, not whether what it says is right.\n\nThe traits:\n{definitions}"
)
RATING_GUIDE = ("Rate how strongly each reply shows each of the five traits, from 1 (not at all) to 5 "
                "(very strongly). Rate only the style of the reply. You are not told which persona, "
                "model or turn a reply comes from.")


def stable_hash(obj, n=8):
    text = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:n]


instructions = JUDGE_INSTRUCTIONS.format(definitions="\n".join(f"- {DEFINITIONS[t]}" for t in TRAITS))
FIELDS = {trait: [1, 2, 3, 4, 5] for trait in TRAITS}
RUBRIC = f"traits@{stable_hash([instructions, FIELDS])}"
JUDGE_SYSTEM = (f"{instructions}\n\nAnswer with one JSON object and nothing else.\n"
                + "\n".join(f'- "{field}": one of {", ".join(map(str, values))}' for field, values in FIELDS.items())
                + "\nExample of the form: {" + ", ".join(f'"{field}": {values[0]}' for field, values in FIELDS.items()) + "}")

print("JUDGE SYSTEM PROMPT:\n" + JUDGE_SYSTEM + "\n")
pd.DataFrame([{"persona": name, "system prompt": text} for name, text in personas.items()])

In [ ]:
THINKING = re.compile(r"<think>.*?</think>", flags=re.DOTALL)


def load_model(key):
    spec = cfg["models"][key]
    tokenizer = AutoTokenizer.from_pretrained(spec["repo"], revision=spec["revision"], **spec.get("tokenizer_kwargs", {}))
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    options = {"revision": spec["revision"]}
    if DEVICE == "cuda":
        dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        options["device_map"] = "auto"
        if spec["precision"] in ("8bit", "4bit"):
            options["quantization_config"] = BitsAndBytesConfig(
                load_in_8bit=spec["precision"] == "8bit", load_in_4bit=spec["precision"] == "4bit",
                bnb_4bit_compute_dtype=dtype)
    else:
        if spec["precision"] != "bf16":
            raise RuntimeError(f"{key}: {spec['precision']} loading needs a CUDA GPU (run it on Colab)")
        dtype = torch.float32
    options["dtype"] = dtype

    print(f"   loading {key} ({spec['repo']}) ...", flush=True)
    try:
        model = AutoModelForCausalLM.from_pretrained(spec["repo"], **options)
    except ValueError:
        model = AutoModelForImageTextToText.from_pretrained(spec["repo"], **options)
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    probe = tokenizer.apply_chat_template([{"role": "system", "content": "SYSTEM-PROBE"}, {"role": "user", "content": "USER-PROBE"}],
                                          tokenize=False, add_generation_prompt=True, **spec.get("chat_template_kwargs", {}))
    assert -1 < probe.find("SYSTEM-PROBE") < probe.find("USER-PROBE"), f"{key}: the chat template does not put the system prompt first"
    return tokenizer, model.eval()


def generate(tokenizer, model, key, conversations, decoding, seed=0):
    global BATCH_SIZE
    template_options = cfg["models"][key].get("chat_template_kwargs", {})
    prompts = [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **template_options)
               for messages in conversations]
    sampling = {"do_sample": True, "temperature": decoding["temperature"], "top_p": decoding["top_p"], "top_k": 0} \
        if decoding["temperature"] > 0 else {"do_sample": False}
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]), reverse=True)
    replies, start = [""] * len(prompts), 0
    while start < len(order):
        batch = order[start:start + BATCH_SIZE]
        encoded = tokenizer([prompts[i] for i in batch], return_tensors="pt", padding=True,
                            add_special_tokens=False).to(model.device)
        torch.manual_seed(seed + start)
        try:
            with torch.inference_mode():
                output = model.generate(**encoded, max_new_tokens=decoding["max_new_tokens"],
                                        pad_token_id=tokenizer.pad_token_id, repetition_penalty=1.0, **sampling)
        except torch.OutOfMemoryError:
            assert BATCH_SIZE > 1, "out of GPU memory with a batch size of 1"
            BATCH_SIZE = max(1, BATCH_SIZE // 2)
            torch.cuda.empty_cache()
            print(f"   out of GPU memory: batch size reduced to {BATCH_SIZE}", flush=True)
            continue
        texts = tokenizer.batch_decode(output[:, encoded["input_ids"].shape[1]:], skip_special_tokens=True)
        for index, text in zip(batch, texts):
            replies[index] = THINKING.sub("", text).strip()
        start += len(batch)
    return replies


def free():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def read_jsonl(path):
    rows = []
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return pd.DataFrame(rows)


def append_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    torn = False
    if path.exists() and path.stat().st_size:
        with open(path, "rb") as f:
            f.seek(-1, os.SEEK_END)
            torn = f.read(1) != b"\n"
    with open(path, "a", encoding="utf-8") as f:
        f.write(("\n" if torn else "") + "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows))


def parse_json(text):
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        value = json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None

In [ ]:
def conversations_for(key):
    revision = cfg["models"][key]["revision"]
    jobs = [{"job_id": f"rq1|{key}|{name}|{d['dialogue_id']}|r{repeat}|" + stable_hash([system, d["user_turns"], CHAT, revision]),
             "persona": name, "dialogue": d["dialogue_id"], "repeat": repeat, "system": system, "user_turns": d["user_turns"]}
            for repeat in range(REPEATS) for d in dialogues for name, system in personas.items()]
    return jobs[:LIMIT] if LIMIT else jobs


def run_model(key, jobs, path):
    stored = read_jsonl(path)
    replies = {} if stored.empty else {(row.job_id, row.turn): row.assistant for row in stored.itertuples()}
    missing = sum((job["job_id"], turn) not in replies for job in jobs for turn in range(1, TURNS + 1))
    print(f"{key}: {len(jobs)} conversations, {len(jobs) * TURNS - missing} replies stored, {missing} to generate", flush=True)
    if not missing:
        return 0
    tokenizer, model = load_model(key)
    for turn in range(1, TURNS + 1):
        todo = [job for job in jobs if (job["job_id"], turn) not in replies]
        for start in range(0, len(todo), CHUNK):
            chunk = todo[start:start + CHUNK]
            conversations = [[{"role": "system", "content": job["system"]},
                              *[message for earlier in range(1, turn) for message in (
                                  {"role": "user", "content": job["user_turns"][earlier - 1]},
                                  {"role": "assistant", "content": replies[(job["job_id"], earlier)]})],
                              {"role": "user", "content": job["user_turns"][turn - 1]}] for job in chunk]
            texts = generate(tokenizer, model, key, conversations, CHAT, int(stable_hash([chunk[0]["job_id"], turn]), 16) % 2**31)
            append_jsonl(path, [{"key": f"{job['job_id']}|t{turn:02d}", "job_id": job["job_id"], "model": key,
                                 "persona": job["persona"], "dialogue": job["dialogue"], "repeat": job["repeat"], "turn": turn,
                                 "user": job["user_turns"][turn - 1], "assistant": text} for job, text in zip(chunk, texts)])
            replies.update({(job["job_id"], turn): text for job, text in zip(chunk, texts)})
            print(f"   turn {turn}: {start + len(chunk)}/{len(todo)}", flush=True)
    del model
    free()
    return missing


frames = []
for key in model_keys:
    path = OUT_DIR / "generations" / f"{key}.jsonl"
    if RERUN and path.exists():
        path.unlink()
    jobs = conversations_for(key)
    started = time.time()
    generated = run_model(key, jobs, path)
    if generated:
        minutes = (time.time() - started) / 60
        every = REPEATS * len(dialogues) * len(personas) * TURNS
        print(f"{key}: {generated} replies in {minutes:.1f} min"
              + (f"; about {minutes / generated * every:.0f} min for all {every} replies" if LIMIT else ""), flush=True)
    stored = read_jsonl(path)
    frames.append(stored[stored["job_id"].isin({job["job_id"] for job in jobs})])

replies = (pd.concat(frames, ignore_index=True).drop_duplicates("key", keep="last")
           .sort_values(["model", "repeat", "dialogue", "persona", "turn"]).reset_index(drop=True))
example = replies[replies["job_id"] == replies["job_id"].iloc[0]]
print(f"\n{len(replies)} replies; one conversation ({example['model'].iloc[0]}, persona {example['persona'].iloc[0]}):")
for row in example.itertuples():
    for speaker, text in ((f"user {row.turn}", row.user), ("reply", row.assistant)):
        print(textwrap.fill(str(text), width=110, initial_indent=f"{speaker:<8}", subsequent_indent=" " * 8))

In [ ]:
HEDGES = ["maybe", "perhaps", "possibly", "probably", "i think", "i guess", "i suppose", "i wonder",
          "sort of", "kind of", "a bit", "a little", "might", "not sure", "i could be wrong", "um",
          "uh", "sorry"]
BOOSTERS = ["obviously", "clearly", "of course", "certainly", "definitely", "undoubtedly", "surely",
            "naturally", "without a doubt", "everyone knows", "as anyone knows", "trivial", "simple"]
AUTOMATIC = {"formal": "formality", "sarcastic": "irony", "warm": "caring", "shy": "hedge_rate",
             "arrogant": "booster_rate"}


def normalize(text):
    text = "".join(" " if char in string.punctuation else char for char in str(text).lower())
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def lexicon_rate(text, terms):
    words = normalize(text).split()
    if not words:
        return 0.0
    haystack = f" {' '.join(words)} "
    return 100.0 * sum(haystack.count(f" {normalize(term)} ") for term in terms) / len(words)


def classify(texts, scorer, batch_size=32):
    device = "cuda" if DEVICE == "cuda" else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(scorer["repo"], revision=scorer["revision"])
    model = AutoModelForSequenceClassification.from_pretrained(scorer["repo"], revision=scorer["revision"]).to(device).eval()
    labels = [str(model.config.id2label[index]).lower() for index in range(model.config.num_labels)]
    blocks = []
    for start in range(0, len(texts), batch_size):
        encoded = tokenizer(texts[start:start + batch_size], return_tensors="pt", padding=True, truncation=True,
                            max_length=512).to(device)
        with torch.inference_mode():
            logits = model(**encoded).logits.float()
        multi_label = model.config.problem_type == "multi_label_classification"
        blocks.append((torch.sigmoid(logits) if multi_label else torch.softmax(logits, dim=-1)).cpu().numpy())
    del model
    free()
    return pd.DataFrame(np.concatenate(blocks), columns=labels)


scorers = cfg["scorers"]
automatic_path = OUT_DIR / "scores" / "automatic.jsonl"
stored = read_jsonl(automatic_path)
new = replies[~replies["key"].isin(set(stored["key"]) if len(stored) else set())]
print(f"automatic measures: {len(replies)} replies, {len(new)} to score", flush=True)
if len(new):
    texts = new["assistant"].tolist()
    columns = {"formality": classify(texts, scorers["formality"])["formal"],
               "irony": classify(texts, scorers["irony"])["irony"],
               "positive": classify(texts, scorers["sentiment"])["positive"],
               "caring": classify(texts, scorers["emotion"])["caring"]}
    append_jsonl(automatic_path, [{"key": key, **{name: float(values.iloc[index]) for name, values in columns.items()},
                                   "hedge_rate": lexicon_rate(text, HEDGES), "booster_rate": lexicon_rate(text, BOOSTERS),
                                   "words": len(text.split())} for index, (key, text) in enumerate(zip(new["key"], texts))])
automatic = read_jsonl(automatic_path).drop_duplicates("key", keep="last")

embeddings_path = OUT_DIR / "scores" / "style_embeddings.npz"
vectors = {}
if embeddings_path.exists():
    saved = np.load(embeddings_path)
    vectors = dict(zip(saved["keys"], saved["vectors"]))
missing = [key for key in replies["key"] if key not in vectors]
if missing:
    from sentence_transformers import SentenceTransformer

    encoder = SentenceTransformer(scorers["style_embedding"]["repo"], revision=scorers["style_embedding"]["revision"])
    texts = replies.set_index("key").loc[missing, "assistant"].tolist()
    vectors.update(zip(missing, encoder.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)))
    np.savez(embeddings_path, keys=np.array(list(vectors), dtype=str), vectors=np.stack(list(vectors.values())))
    del encoder
    free()
automatic.describe().round(3)

In [ ]:
def verdict_file(stage):
    return OUT_DIR / "annotation" / stage / f"judge_check_{JUDGE}.txt"


def judge_verdict():
    for stage in ("final", "pilot"):
        verdict = verdict_file(stage).read_text(encoding="utf-8").strip() if verdict_file(stage).exists() else ""
        if verdict:
            return verdict
    return ""


def valid_answer(answer):
    if answer is None:
        return None
    cleaned = {}
    for field, allowed in FIELDS.items():
        value = answer.get(field)
        if isinstance(value, str) and value.strip().isdigit():
            value = int(value)
        if isinstance(value, float) and value.is_integer():
            value = int(value)
        if isinstance(value, bool) or value not in allowed:
            return None
        cleaned[field] = value
    return cleaned


judge_path = OUT_DIR / "scores" / "judge_traits.jsonl"
stored = read_jsonl(judge_path)
done = set(stored["key"]) if len(stored) else set()
items = replies.assign(text="User message:\n" + replies["user"] + "\n\nReply to rate:\n" + replies["assistant"])
todo = items[[f"{key}|{RUBRIC}|{JUDGE}" not in done for key in items["key"]]]
judge_may_run = not study_run or bool(LIMIT) or not CHECK.get("required", True) or judge_verdict() == "PASS"
print(f"judge {JUDGE}: {len(items)} replies, {len(items) - len(todo)} rated, {len(todo)} to rate", flush=True)
if len(todo) and not judge_may_run:
    print("   judge not run: check it on a sample first (LIMIT, then the rating cell), "
          "or set judge_check.required to false in config.yaml")
elif len(todo):
    tokenizer, model = load_model(JUDGE)
    for start in range(0, len(todo), CHUNK):
        chunk = todo.iloc[start:start + CHUNK]
        texts = generate(tokenizer, model, JUDGE, [[{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": text}]
                                                   for text in chunk["text"]], JUDGING, seed=start)
        rows = []
        for key, text in zip(chunk["key"], texts):
            answer = valid_answer(parse_json(text))
            rows.append({"key": f"{key}|{RUBRIC}|{JUDGE}", "item": key, "rubric": RUBRIC, "judge": JUDGE,
                         "valid": answer is not None, **(answer or dict.fromkeys(FIELDS)), "raw": text})
        append_jsonl(judge_path, rows)
        print(f"   rated {start + len(chunk)}/{len(todo)}", flush=True)
    del tokenizer, model
    free()

judged = read_jsonl(judge_path)
if len(judged):
    judged = judged[(judged["judge"] == JUDGE) & (judged["rubric"] == RUBRIC) & judged["item"].isin(set(replies["key"]))]
    print(f"   readable answers: {judged['valid'].mean():.0%} of {len(judged)}")
    judged = judged[judged["valid"]].drop_duplicates("item", keep="last")

table = replies.merge(automatic, on="key", how="left")
if len(judged):
    table = table.merge(judged[["item", *TRAITS]].rename(columns={"item": "key"}), on="key", how="left")
else:
    table = table.assign(**{trait: np.nan for trait in TRAITS})
table["adherence"] = [row[row["persona"]] if row["persona"] in TRAITS else np.nan for _, row in table.iterrows()]
first = {row.job_id: vectors[row.key] for row in table[table["turn"] == 1].itertuples() if row.key in vectors}
table["style_similarity_to_first_reply"] = [float(vectors[row.key] @ first[row.job_id])
                                            if row.key in vectors and row.job_id in first else np.nan for row in table.itertuples()]
table[["model", "persona", "dialogue", "turn", *TRAITS, "adherence", *AUTOMATIC.values()]].head(8)

In [ ]:
rating_stage = "pilot" if (LIMIT or not study_run) else "final"
RATINGS_DIR = OUT_DIR / "annotation" / rating_stage


def export_for_rating(items):
    if (RATINGS_DIR / "items.csv").exists():
        foreign = ~pd.read_csv(RATINGS_DIR / "key.csv")["key"].isin(set(items["key"]))
        if foreign.any():
            print(f"WARNING: {foreign.sum()} of the items in {RATINGS_DIR / 'items.csv'} are not among this run's "
                  "replies (written by an earlier run). Delete the folder to write a new sample, unless raters are "
                  "already working on it.")
        return
    RATINGS_DIR.mkdir(parents=True, exist_ok=True)
    n = CHECK["pilot_items"] if rating_stage == "pilot" else cfg["rq1"]["annotation"]["n_items"]
    groups = [group for _, group in items.groupby(["persona", "model"], sort=True)]
    picked = pd.concat([group.sample(n=min(math.ceil(n / len(groups)), len(group)), random_state=SEED) for group in groups])
    picked = picked.sample(n=min(n, len(picked)), random_state=SEED).reset_index(drop=True)
    picked.insert(0, "item_id", [f"item{index + 1:03d}" for index in range(len(picked))])
    picked[["item_id", "user", "assistant"]].assign(**{trait: "" for trait in TRAITS}).to_csv(RATINGS_DIR / "items.csv", index=False)
    picked[["item_id", "key"]].to_csv(RATINGS_DIR / "key.csv", index=False)
    lines = [RATING_GUIDE, "", "Scales:", *[f"- {t}: {DEFINITIONS[t]} Rate 1 to 5." for t in TRAITS], "",
             "Save your filled copy of items.csv as ratings_<your name>.csv in this folder."]
    (RATINGS_DIR / "guide.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"wrote {len(picked)} items to rate: {RATINGS_DIR / 'items.csv'}")


def load_ratings():
    frames = []
    for path in sorted(RATINGS_DIR.glob("ratings_*.csv")):
        with open(path, encoding="utf-8-sig") as f:
            header = f.readline()
        sheet = pd.read_csv(path, sep=";" if header.count(";") > header.count(",") else ",", encoding="utf-8-sig")
        sheet = sheet[["item_id", *TRAITS]].merge(pd.read_csv(RATINGS_DIR / "key.csv"), on="item_id")
        frames.append(sheet.assign(rater=path.stem.removeprefix("ratings_")))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def kappa(first, second, weights):
    from sklearn.metrics import cohen_kappa_score

    return np.nan if len(set(first) | set(second)) < 2 else float(cohen_kappa_score(first, second, weights=weights))


def agreement(ratings, machine):
    import krippendorff

    rows = []
    for scale in TRAITS:
        wide = ratings.pivot_table(index="key", columns="rater", values=scale, aggfunc="first").dropna()
        numeric = pd.api.types.is_numeric_dtype(ratings[scale])
        weights = "quadratic" if numeric else None
        row = {"scale": scale, "items": len(wide), "raters": wide.shape[1],
               "alpha_humans": np.nan, "kappa_humans": np.nan, "kappa_judge": np.nan, "verdict": ""}
        if len(wide) >= 2 and wide.shape[1] >= 2:
            categories = pd.Index(pd.unique(wide.to_numpy().ravel()))
            codes = wide if numeric else wide.apply(categories.get_indexer)
            row["alpha_humans"] = krippendorff.alpha(reliability_data=codes.to_numpy(dtype=float).T,
                                                     level_of_measurement="ordinal" if numeric else "nominal")
            pairs = [(a, b) for index, a in enumerate(wide.columns) for b in wide.columns[index + 1:]]
            row["kappa_humans"] = float(np.mean([kappa(wide[a], wide[b], weights) for a, b in pairs]))
        if len(wide):
            consensus = wide.median(axis=1).round().astype(int) if numeric else wide.mode(axis=1)[0]
            both = pd.DataFrame({"human": consensus}).join(machine.set_index("key")[scale].rename("judge"), how="inner").dropna()
            if len(both) >= 2:
                row["kappa_judge"] = kappa(both["human"], both["judge"].astype(int) if numeric else both["judge"], weights)
        if not np.isnan(row["kappa_humans"]) and row["kappa_humans"] < CHECK["min_kappa"]:
            row["verdict"] = "FIX RUBRIC"
        elif not np.isnan(row["kappa_judge"]):
            needed = CHECK["min_kappa"] if np.isnan(row["kappa_humans"]) else max(CHECK["min_kappa"], row["kappa_humans"] - CHECK["tolerance"])
            row["verdict"] = "PASS" if row["kappa_judge"] >= needed else "FAIL"
        rows.append(row)
    return pd.DataFrame(rows)


def validate_measures(ratings, automatic, pairs):
    from scipy.stats import spearmanr

    rows = []
    for measure, scale in pairs.items():
        both = pd.DataFrame({"human": ratings.groupby("key")[scale].median()}).join(
            automatic.set_index("key")[measure], how="inner").dropna()
        varies = both["human"].nunique() > 1 and both[measure].nunique() > 1
        rho = spearmanr(both["human"], both[measure]).statistic if len(both) >= 3 and varies else np.nan
        rows.append({"measure": measure, "human_scale": scale, "items": len(both), "spearman": rho,
                     "tracks_humans": bool(rho >= cfg["validation"]["min_spearman"]) if not np.isnan(rho) else None})
    return pd.DataFrame(rows)


tables = {}
export_for_rating(table)
ratings = load_ratings()
print("items to rate :", RATINGS_DIR / "items.csv")
print("ratings found :", sorted(path.name for path in RATINGS_DIR.glob("ratings_*.csv")) or "none yet")
if len(ratings):
    tables["agreement"] = agreement(ratings, table[["key", *TRAITS]])
    found = set(tables["agreement"]["verdict"]) - {""}
    verdict = next((v for v in ("FIX RUBRIC", "FAIL", "PASS") if v in found), "")
    verdict_file(rating_stage).write_text(verdict + "\n", encoding="utf-8")
    tables["measure_validation"] = validate_measures(ratings, table, {
        "formality": "formal", "irony": "sarcastic", "positive": "warm", "caring": "warm",
        "hedge_rate": "shy", "booster_rate": "arrogant"})
    print("\nHUMAN RATINGS\n" + tables["agreement"].round(3).to_string(index=False))
    print("\nAUTOMATIC MEASURES AGAINST THE HUMAN MEDIAN\n" + tables["measure_validation"].round(3).to_string(index=False))
print("\nverdict of the judge check:", judge_verdict() or "none yet")

In [ ]:
def mean_ci(values, clusters=None):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return {"mean": np.nan, "lo": np.nan, "hi": np.nan, "n": 0}
    codes = np.arange(len(values)) if clusters is None else pd.factorize(np.asarray(clusters))[0]
    sums, counts = np.bincount(codes, weights=values), np.bincount(codes)
    rng = np.random.default_rng(SEED)
    means = []
    for start in range(0, N_BOOTSTRAP, 200):
        draws = rng.integers(0, len(sums), size=(min(200, N_BOOTSTRAP - start), len(sums)))
        means.append(sums[draws].sum(axis=1) / counts[draws].sum(axis=1))
    lo, hi = np.percentile(np.concatenate(means), [2.5, 97.5])
    return {"mean": float(values.mean()), "lo": float(lo), "hi": float(hi), "n": int(len(values))}


def summarize(part, value, by, cluster=None):
    return pd.DataFrame([{**dict(zip(by, key if isinstance(key, tuple) else (key,))),
                          **mean_ci(group[value], group[cluster] if cluster else None)}
                         for key, group in part.groupby(by, sort=True)])


def conversation_slopes(part, value):
    rows = []
    for job_id, conversation in part.dropna(subset=[value]).groupby("job_id"):
        if conversation["turn"].nunique() >= 2:
            rows.append({"job_id": job_id, "model": conversation["model"].iloc[0], "persona": conversation["persona"].iloc[0],
                         "slope": np.polyfit(conversation["turn"], conversation[value].astype(float), 1)[0]})
    return pd.DataFrame(rows, columns=["job_id", "model", "persona", "slope"])


note = "" if judge_verdict() == "PASS" else " [judge not validated]"
rated = table[table["adherence"].notna()].copy()
if len(rated) < 4:
    reason = ("no usable judge ratings yet (the judge has not run, or its answers could not be read)" if rated.empty
              else f"too few judge ratings for the main test: {len(rated)} rated replies of a trait persona")
    primary = [{"result": reason, "mean": np.nan, "lo": np.nan, "hi": np.nan, "p_value": np.nan, "replies": len(rated)}]
else:
    rated["adherence"] = rated["adherence"].astype(float)
    level = mean_ci(rated["adherence"], rated["job_id"])
    mean_rating = {"result": f"mean rating of the assigned trait (1-5){note}", "mean": level["mean"], "lo": level["lo"],
                   "hi": level["hi"], "p_value": np.nan, "replies": len(rated)}
    if rated["adherence"].nunique() == 1:
        primary = [{"result": f"MAIN TEST: change of the rating per turn (every reply has the same rating, so nothing can be tested){note}",
                    "mean": 0.0, "lo": np.nan, "hi": np.nan, "p_value": np.nan, "replies": len(rated)}, mean_rating]
    else:
        try:
            import statsmodels.formula.api as smf

            formula = "adherence ~ turn" + (" + C(persona)" if rated["persona"].nunique() > 1 else "") \
                      + (" + C(model)" if rated["model"].nunique() > 1 else "")
            fit = smf.mixedlm(formula, rated, groups=rated["job_id"]).fit()
            lo, hi = fit.conf_int().loc["turn"]
            primary = [{"result": f"MAIN TEST: change of the rating per turn (mixed model, random intercept per conversation){note}",
                        "mean": fit.params["turn"], "lo": lo, "hi": hi, "p_value": fit.pvalues["turn"], "replies": len(rated)}]
        except Exception as error:
            from scipy.stats import ttest_1samp

            slopes = conversation_slopes(rated, "adherence")["slope"].to_numpy()
            stats = mean_ci(slopes)
            primary = [{"result": f"MAIN TEST: mean slope per conversation (mixed model not fitted: {type(error).__name__}){note}",
                        "mean": stats["mean"], "lo": stats["lo"], "hi": stats["hi"],
                        "p_value": ttest_1samp(slopes, 0).pvalue if len(slopes) > 1 else np.nan, "replies": len(rated)}]
        primary.append(mean_rating)
tables["primary_result"] = pd.DataFrame(primary)
print("MAIN RESULT (RQ1)\n" + tables["primary_result"].round(4).to_string(index=False))

In [ ]:
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
TEXT, MUTED, GRID, SURFACE = "#52514e", "#898781", "#e1e0d9", "#fcfcfb"


def line_chart(part, x, y="mean", hue=None, panel=None, title="", ylabel="", sharey=True):
    panels = list(dict.fromkeys(part[panel])) if panel else [None]
    columns = len(panels) if len(panels) <= 5 else 4
    rows = math.ceil(len(panels) / columns)
    figure, axes = plt.subplots(rows, columns, figsize=(4 * columns, 3.2 * rows), squeeze=False, sharey=sharey,
                                facecolor=SURFACE)
    groups = list(dict.fromkeys(part[hue])) if hue else [None]
    for axis, name in zip(axes.flat, panels):
        shown = part if name is None else part[part[panel] == name]
        for index, group in enumerate(groups):
            line = (shown if group is None else shown[shown[hue] == group]).sort_values(x)
            color = SERIES[index % len(SERIES)]
            axis.plot(line[x], line[y], marker="o", markersize=6, linewidth=2, color=color,
                      label=None if group is None else str(group))
            if {"lo", "hi"} <= set(line.columns):
                axis.fill_between(line[x], line["lo"], line["hi"], alpha=0.1, color=color, linewidth=0)
        axis.set_title(str(name) if name is not None else title, fontsize=10, loc="left", color=TEXT)
        axis.set_xlabel(x)
        axis.set_xticks(sorted(set(shown[x])))
        axis.set_facecolor(SURFACE)
        axis.grid(axis="y", color=GRID, linewidth=1)
        axis.set_axisbelow(True)
        axis.tick_params(colors=MUTED, labelsize=8, length=0)
        axis.xaxis.label.set_color(TEXT)
        axis.yaxis.label.set_color(TEXT)
        for side in ("top", "right", "left"):
            axis.spines[side].set_visible(False)
        axis.spines["bottom"].set_color(GRID)
    for axis in axes[:, 0]:
        axis.set_ylabel(ylabel or y)
    for axis in list(axes.flat)[len(panels):]:
        axis.set_visible(False)
    if panel and title:
        figure.suptitle(title, x=0.01, ha="left", fontsize=11, color=TEXT)
    figure.tight_layout()
    if hue:
        figure.legend(*axes.flat[0].get_legend_handles_labels(), title=hue, fontsize=8, title_fontsize=8,
                      frameon=False, loc="center left", bbox_to_anchor=(1.0, 0.5), labelcolor=TEXT)
    return figure


figures = {}
if table["adherence"].notna().any():
    tables["adherence_by_turn"] = summarize(table[table["adherence"].notna()], "adherence", ["model", "persona", "turn"], "job_id")
    figures["adherence_by_turn"] = line_chart(tables["adherence_by_turn"], "turn", hue="persona", panel="model",
                                              title="Judge rating of the assigned trait, by turn", ylabel="rating (1-5)")

rows = []
for trait in TRAITS:
    for model, part in table[table["persona"].isin([trait, "neutral"])].dropna(subset=[trait]).groupby("model"):
        own, neutral = part[part["persona"] == trait], part[part["persona"] == "neutral"]
        if len(own) and len(neutral):
            with_persona, without = mean_ci(own[trait], own["job_id"]), mean_ci(neutral[trait], neutral["job_id"])
            rows.append({"model": model, "trait": trait, "with_persona": with_persona["mean"], "lo": with_persona["lo"],
                         "hi": with_persona["hi"], "neutral": without["mean"], "neutral_lo": without["lo"],
                         "neutral_hi": without["hi"], "difference": with_persona["mean"] - without["mean"]})
if rows:
    tables["rating_vs_neutral"] = pd.DataFrame(rows)

parts = [summarize(table[table["persona"].isin([trait, "neutral"])], measure, ["model", "persona", "turn"], "job_id")
         .assign(trait=trait, measure=measure) for trait, measure in AUTOMATIC.items() if table["persona"].isin([trait, "neutral"]).any()]
if parts:
    tables["automatic_by_turn"] = pd.concat(parts, ignore_index=True)
    own = tables["automatic_by_turn"][tables["automatic_by_turn"]["persona"] == tables["automatic_by_turn"]["trait"]]
    if len(own):
        figures["automatic_by_turn"] = line_chart(own.assign(panel=own["persona"] + ": " + own["measure"]), "turn", hue="model",
                                                  panel="panel", sharey=False, ylabel="score",
                                                  title="Automatic measure of each persona's trait, by turn")

rows = []
for (model, persona), part in table[table["persona"].isin(TRAITS)].groupby(["model", "persona"]):
    for value in ["adherence", AUTOMATIC[persona]]:
        slopes = conversation_slopes(part, value)["slope"].to_numpy()
        if len(slopes):
            stats = mean_ci(slopes)
            rows.append({"model": model, "persona": persona, "measure": value, "slope_per_turn": stats["mean"],
                         "lo": stats["lo"], "hi": stats["hi"], "conversations": stats["n"]})
tables["drift_slopes"] = pd.DataFrame(rows)

tables["style_similarity_by_turn"] = summarize(table[table["turn"] > 1], "style_similarity_to_first_reply",
                                               ["model", "persona", "turn"], "job_id")
from sklearn.linear_model import LogisticRegression

rows = []
for model, part in table[table["key"].isin(vectors)].groupby("model"):
    early, late = part[part["turn"] <= 2], part[part["turn"] > 2]
    if early["persona"].nunique() < 2 or late.empty:
        continue
    classifier = LogisticRegression(max_iter=1000).fit(np.stack([vectors[k] for k in early["key"]]), early["persona"])
    for turn, at_turn in late.groupby("turn"):
        predicted = classifier.predict(np.stack([vectors[k] for k in at_turn["key"]]))
        rows.append({"model": model, "turn": turn, "accuracy": float((predicted == at_turn["persona"]).mean()),
                     "chance": 1 / early["persona"].nunique(), "replies": len(at_turn)})
if rows:
    tables["identifiability_by_turn"] = pd.DataFrame(rows)
    figures["identifiability_by_turn"] = line_chart(tables["identifiability_by_turn"], "turn", "accuracy", hue="model",
                                                    title="Can a classifier still tell the personas apart?", ylabel="accuracy")

for name in ("rating_vs_neutral", "drift_slopes"):
    if name in tables:
        print(f"{name.upper()}\n" + tables[name].round(3).to_string(index=False) + "\n")

In [ ]:
for by in ("model", "persona"):
    parts = [summarize(part.dropna(subset=[value]), value, [by], cluster).assign(measure=name)
             for name, part, value, cluster in [
                 (f"trait rating (1-5){note}", table, "adherence", "job_id"),
                 (f"rating change per turn{note}", conversation_slopes(table, "adherence"), "slope", None),
                 ("style similarity to the first reply", table[table["turn"] > 1], "style_similarity_to_first_reply", "job_id")]
             if by in part and part[value].notna().any()]
    columns = ["measure", by, "mean", "lo", "hi", "n"]
    tables[f"by_{by}"] = pd.concat(parts, ignore_index=True)[columns] if parts else pd.DataFrame(columns=columns)
tables["turn_scores"] = table.drop(columns=["user", "assistant"])

(OUT_DIR / "tables").mkdir(exist_ok=True)
(OUT_DIR / "figures").mkdir(exist_ok=True)
for name, content in tables.items():
    content.to_csv(OUT_DIR / "tables" / f"{name}.csv", index=False)
for name, figure in figures.items():
    figure.savefig(OUT_DIR / "figures" / f"{name}.png", dpi=130, bbox_inches="tight")
(OUT_DIR / "run_info.json").write_text(json.dumps({
    "models": model_keys, "judge": JUDGE, "personas": personas, "dialogues": [d["dialogue_id"] for d in dialogues],
    "turns": TURNS, "repeats": REPEATS, "limit": LIMIT, "decoding": CHAT, "rubric": RUBRIC,
    "model_specs": {k: cfg["models"][k] for k in [*model_keys, JUDGE]}, "device": DEVICE, "torch": torch.__version__,
}, indent=2))
print("saved to", OUT_DIR)
print("tables:", ", ".join(tables))